# Cross-dataset generalization

Train on one source, test on the other.

In [ ]:
from pathlib import Path
import common
common.ensure_dirs()
print("Working directory:", Path.cwd())

import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

### 1. Setup (load the model etc.)

In [ ]:
model = common.load_model(prefer_bridge=True)

### 2. Load the sources

In [ ]:
with open("data/roleplay_generated.json") as f:
    roleplay_data = json.load(f)
with open("data/instructed_generated.json") as f:
    instructed_data = json.load(f)

for d in roleplay_data: d["source"] = "roleplay"
for d in instructed_data: d["source"] = "instructed"

print(f"Roleplay: {len(roleplay_data)}, Instructed: {len(instructed_data)}")

### 3. Split each source independently into train/eval

In [ ]:
def split_by_label(data, test_size=0.3, seed=42):
    labels = [1 if d["label"] == "deceptive" else 0 for d in data]
    train, eval_ = train_test_split(data, test_size=test_size, stratify=labels, random_state=seed)
    return train, eval_

In [ ]:
roleplay_train, roleplay_eval = split_by_label(roleplay_data)
instructed_train, instructed_eval = split_by_label(instructed_data)

print(f"Roleplay train/eval: {len(roleplay_train)}/{len(roleplay_eval)}")
print(f"Instructed train/eval: {len(instructed_train)}/{len(instructed_eval)}")

### 4. Feature extraction

In [ ]:
def extract_and_cache(model, data, layers, name):
    X_by_layer, y = build_features_multilayer(model, data, layers)
    np.savez(f"data/features_{name}.npz", y=y, **{f"X_layer{l}": X_by_layer[l] for l in layers})
    return X_by_layer, y

In [ ]:
n_layers = model.cfg.n_layers
candidate_layers = list(range(2, n_layers, 3))

X_roleplay_train, y_roleplay_train = extract_and_cache(model, roleplay_train, candidate_layers, "roleplay_train")
X_roleplay_eval, y_roleplay_eval = extract_and_cache(model, roleplay_eval, candidate_layers, "roleplay_eval")
X_instructed_train, y_instructed_train = extract_and_cache(model, instructed_train, candidate_layers, "instructed_train")
X_instructed_eval, y_instructed_eval = extract_and_cache(model, instructed_eval, candidate_layers, "instructed_eval")

### 5. Cross-data generalization main function

In [ ]:
def cross_eval(X_train_src, y_train_src, eval_sets, layer, train_name):
    clf = LogisticRegression(max_iter=1000, C=10.0)
    clf.fit(X_train_src[layer], y_train_src)

    results = {}
    for eval_name, (X_eval_src, y_eval_src) in eval_sets.items():
        scores = clf.predict_proba(X_eval_src[layer])[:, 1]
        auroc = roc_auc_score(y_eval_src, scores)
        results[eval_name] = auroc
        print(f"  train={train_name}, eval={eval_name}, layer={layer}: AUROC={auroc:.3f}")
    return results

In [ ]:
eval_sets = {
    "roleplay": (X_roleplay_eval, y_roleplay_eval),
    "instructed": (X_instructed_eval, y_instructed_eval),
}

cross_results = {}
for layer in candidate_layers:
    print(f"--- layer {layer} ---")
    cross_results[("roleplay", layer)] = cross_eval(X_roleplay_train, y_roleplay_train, eval_sets, layer, "roleplay")
    cross_results[("instructed", layer)] = cross_eval(X_instructed_train, y_instructed_train, eval_sets, layer, "instructed")

In [ ]:
# Save results to JSON file
results_table = []
for (train_src, layer), evals in cross_results.items():
    for eval_src, auroc in evals.items():
        results_table.append({"train": train_src, "eval": eval_src, "layer": layer, "auroc": auroc})

with open("results/cross_generalization.json", "w") as f:
    json.dump(results_table, f, indent=2)

### 6. Plotting